## Weighted by Time Only Embeddings Approach

In [ ]:
# core data handling
import pandas as pd
import numpy as np
# text processing
import re
# embeddings
from sentence_transformers import SentenceTransformer
# download files
from google.colab import files

Section 1: Import Data and Load

In [ ]:
# Section 1: Prepare the review-level data so it’s clean, consistent, and ready for embeddings and weighting
df_raw = pd.read_csv("/content/wealthtender_reviews_clean.csv")

# Parse review_date to compute recency later
df_raw["review_date"] = pd.to_datetime(df_raw["review_date"], errors="coerce")
if df_raw["review_date"].isna().any():
    bad = df_raw.loc[df_raw["review_date"].isna(), ["ID", "review_date"]].head(10)
    raise ValueError(f"Some review_date values could not be parsed. Examples:\n{bad}")

# Rename columns to a consistent schema for later in the pipeline
df = df_raw.rename(columns={
    "ID": "review_id",
    "review_text_clean": "review_text_clean2",
    "token_count": "n_tokens",
    "clean_token_count": "n_tokens_nostop",
}).copy()

# Lock the clean text column and drop any empty rows
df["review_text_clean2"] = df["review_text_clean2"].astype(str)
df = df[df["review_text_clean2"].str.strip().ne("")].reset_index(drop=True)

# Compute age_years relative to the newest review so the weighting is reproducible
ref_date = df["review_date"].max()
df["age_years"] = ((ref_date - df["review_date"]).dt.total_seconds() / (365.25 * 24 * 3600)).clip(lower=0)

# Checking here to make sure everything looks correct
print(f"Loaded df shape: {df.shape}")
print(f"Reference date used for age_years: {ref_date.date()}")
print("Min/median/max age_years:", float(df["age_years"].min()), float(df["age_years"].median()), float(df["age_years"].max()))
df[["review_id","advisor_id","advisor_name","review_date","age_years","n_tokens","n_tokens_nostop"]].head(3)

Loaded df shape: (4731, 26)
Reference date used for age_years: 2026-01-07
Min/median/max age_years: 0.0 1.54919575633128 17.11946725986767


,review_id,advisor_id,advisor_name,review_date,age_years,n_tokens,n_tokens_nostop
0,55476,https://wealthtender.com/financial-advisors/om...,"Omar A. Morillo, CFP®, ChFC®, AIF®",2026-01-07 15:15:00,0.000000,93,93
1,55446,https://wealthtender.com/financial-advisors/om...,"Omar A. Morillo, CFP®, ChFC®, AIF®",2026-01-06 20:44:00,0.002112,47,47
2,55445,https://wealthtender.com/financial-advisors/om...,"Omar A. Morillo, CFP®, ChFC®, AIF®",2026-01-06 20:44:00,0.002112,142,142


Section 2: Review-level Embeddings

In [ ]:
# Section 2: Create review-level embeddings

# Flag whether the advisor name shows up in the review text
def contains_advisor_name_robust(text, name):
    if not isinstance(text, str) or not isinstance(name, str):
        return 0
    text_l = text.lower()
    name_norm = " ".join(name.lower().split())
    if not name_norm:
        return 0
    if re.search(rf"\b{re.escape(name_norm)}\b", text_l):
        return 1
    first = name_norm.split()[0] if name_norm.split() else ""
    if len(first) >= 2 and re.search(rf"\b{re.escape(first)}\b", text_l):
        return 1
    return 0

df["contains_advisor_name"] = df.apply(lambda r: contains_advisor_name_robust(r["review_text_clean2"], r["advisor_name"]), axis=1)

# Remove advisor name only at embedding-time so names don’t dominate the vector, but keep original text unchanged
def strip_advisor_name_for_embedding(text, name):
    if not isinstance(text, str) or not isinstance(name, str) or not name.strip():
        return text
    pattern = re.compile(re.escape(name.strip()), re.IGNORECASE)
    cleaned = pattern.sub(" ", text)
    return re.sub(r"\s+", " ", cleaned).strip()

df["text_for_embedding"] = df.apply(lambda r: strip_advisor_name_for_embedding(r["review_text_clean2"], r["advisor_name"]), axis=1)

# Embed each review using the agreed model, for now we are using (MiniLM-L6-v2 outputs 384-d vectors)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(df["text_for_embedding"].tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=False)

# Store embeddings as lists
df["embedding"] = embeddings.tolist()

print("embedding dim:", len(df["embedding"].iloc[0]))
df[["advisor_name", "contains_advisor_name", "review_text_clean2", "text_for_embedding"]].head(3)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/74 [00:00<?, ?it/s]

embedding dim: 384


,advisor_name,contains_advisor_name,review_text_clean2,text_for_embedding
0,"Omar A. Morillo, CFP®, ChFC®, AIF®",1,omar morillo is an exceptional wealth advisor ...,omar morillo is an exceptional wealth advisor ...
1,"Omar A. Morillo, CFP®, ChFC®, AIF®",1,omar is an exceptional advisor. beyond the out...,omar is an exceptional advisor. beyond the out...
2,"Omar A. Morillo, CFP®, ChFC®, AIF®",1,we are extremely happy with the work omar mori...,we are extremely happy with the work omar mori...


Section 3: Defining Weights (Recency + Informativeness)

In [ ]:
# Section 3: Compute review-level weights so more recent and more informative reviews matter more in aggregation

# Apply recency weighting using exponential decay (newer reviews carry more weight)
HALF_LIFE_YEARS = 2.0
df["w_time"] = 0.5 ** (df["age_years"] / HALF_LIFE_YEARS)

# Apply informativeness weighting based on review length (capped so very long reviews don’t dominate)
TOKEN_REF = 80
INFO_CAP = 2.0
INFO_FLOOR = 0.2

df["w_info"] = np.sqrt(df["n_tokens_nostop"].clip(lower=0) / TOKEN_REF)
df["w_info"] = df["w_info"].clip(lower=INFO_FLOOR, upper=INFO_CAP)

# Final weights to use later
df["review_weight_time"] = df["w_time"]
df["review_weight_time_info"] = df["w_time"] * df["w_info"]

# Check to make sure the weighting behaves as expected
print("w_time min/med/max:", float(df["w_time"].min()), float(df["w_time"].median()), float(df["w_time"].max()))
print("w_info min/med/max:", float(df["w_info"].min()), float(df["w_info"].median()), float(df["w_info"].max()))
print("w_time_info min/med/max:", float(df["review_weight_time_info"].min()), float(df["review_weight_time_info"].median()), float(df["review_weight_time_info"].max()))

w_time min/med/max: 0.002650107080849136 0.5845515334000478 1.0
w_info min/med/max: 0.2 0.8944271909999159 2.0
w_time_info min/med/max: 0.002388774241367938 0.5070556640023016 1.9958503829040668


Section 4: Advisor-level Aggregration

In [ ]:
# Section 4: Aggregate review embeddings to advisor-level using time-only weights

def weighted_mean(E, w):
    w = np.asarray(w, dtype=float)
    denom = w.sum()
    if denom <= 0:
        return E.mean(axis=0)
    return (E * w[:, None]).sum(axis=0) / denom

def effective_n(w):
    w = np.asarray(w, dtype=float)
    s2 = (w**2).sum()
    if s2 == 0:
        return 0.0
    return float((w.sum() ** 2) / s2)

advisor_rows = []
for advisor_id, g in df.groupby("advisor_id"):
    # stack review embeddings for this advisor to compute a single advisor representation
    E = np.vstack(g["embedding"].values)

    # compute the advisor embedding as a time-weighted average of review embeddings
    emb_w_time = weighted_mean(E, g["review_weight_time"].values)

    # saved diagnostics so scoring can understand embedding reliability
    advisor_rows.append({"advisor_id": advisor_id, "advisor_name": g["advisor_name"].iloc[0], "advisor_embedding_weighted_time": emb_w_time, "n_reviews_used": int(len(g)), "total_tokens": int(g["n_tokens"].sum()), "total_tokens_nostop": int(g["n_tokens_nostop"].sum()), "median_age_years": float(g["age_years"].median()), "min_review_date": g["review_date"].min(), "max_review_date": g["review_date"].max(), "effective_n_time": effective_n(g["review_weight_time"].values)})

df_advisors = pd.DataFrame(advisor_rows)

print("df_advisors shape:", df_advisors.shape)
print("embedding dim:", len(df_advisors["advisor_embedding_weighted_time"].iloc[0]))
df_advisors.head(3)

df_advisors shape: (336, 10)
embedding dim: 384


,advisor_id,advisor_name,advisor_embedding_weighted_time,n_reviews_used,total_tokens,total_tokens_nostop,median_age_years,min_review_date,max_review_date,effective_n_time
0,Press Advisor Test,Press Advisor Test,"[-0.07958291471004486, 0.0627366304397583, -0....",1,6,6,4.774452,2021-03-30 18:24:00,2021-03-30 18:24:00,1.000000
1,https://wealthtender.com/advisory-firms/abundo...,Abundo Wealth,"[-0.005624547706825571, -0.0009465943245273385...",37,3492,3492,1.915497,2022-05-05 17:37:00,2025-08-03 17:44:00,34.409795
2,https://wealthtender.com/advisory-firms/allegi...,"Allegiance Financial Group Advisory Services, LLC","[-0.025489320718860495, -0.002758181935373463,...",12,723,723,0.556037,2025-06-04 19:31:00,2025-10-14 13:33:00,11.976141


* Ask Carolina about these Press Advisor Test reviews, legitimate?

In [ ]:
(df["advisor_name"] == "Press Advisor Test").sum()

np.int64(2)

In [ ]:
(df["advisor_id"] == "Press Advisor Test").sum()

np.int64(1)

In [ ]:
df[df["advisor_name"] == "Press Advisor Test"][
    ["review_id", "advisor_id", "review_date", "n_tokens", "age_years"]]

,review_id,advisor_id,review_date,n_tokens,age_years
4573,15822,https://wealthtender.com/financial-advisors/pr...,2021-03-30 19:29:00,6,4.774329
4574,15819,Press Advisor Test,2021-03-30 18:24:00,6,4.774452


Section 5: Save Deliverables

In [ ]:
# Section 6: review-level base + advisor time-weighted embeddings

df_reviews_embedded = df[[
    "review_id",
    "advisor_id",
    "advisor_name",
    "review_date",
    "age_years",
    "n_tokens",
    "n_tokens_nostop",
    "review_text_clean2",
    "contains_advisor_name",
    "embedding",
    "w_time",
    "w_info",
    "review_weight_time",
    "review_weight_time_info"]].copy()

df_reviews_embedded["embedding"] = df_reviews_embedded[
    "embedding"].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else x)

# Prepare the advisor-level dataset with time-weighted embeddings
df_advisors_weighted_time = df_advisors[[
    "advisor_id","advisor_name",
    "advisor_embedding_weighted_time",
    "n_reviews_used","total_tokens",
    "total_tokens_nostop",
    "median_age_years",
    "min_review_date",
    "max_review_date",
    "effective_n_time"]].copy()

df_advisors_weighted_time["advisor_embedding_weighted_time"] = df_advisors_weighted_time[
    "advisor_embedding_weighted_time"].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else x)

# download parquet and csv, ask which file is preferred
df_reviews_embedded.to_parquet("df_reviews_embedded.parquet", index=False)
df_reviews_embedded.to_csv("df_reviews_embedded.csv", index=False)

df_advisors_weighted_time.to_parquet("df_advisors_weighted_time.parquet", index=False)
df_advisors_weighted_time.to_csv("df_advisors_weighted_time.csv", index=False)

print("Saved deliverables:")
print("df_reviews_embedded.parquet / df_reviews_embedded.csv")
print("df_advisors_weighted_time.parquet / df_advisors_weighted_time.csv")

Saved deliverables:
df_reviews_embedded.parquet / df_reviews_embedded.csv
df_advisors_weighted_time.parquet / df_advisors_weighted_time.csv


In [ ]:
# bundle deliverables into a zip

import zipfile
from google.colab import files

zip_path = "advisor_embeddings_time_weighted_deliverables.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write("df_reviews_embedded.parquet")
    z.write("df_reviews_embedded.csv")
    z.write("df_advisors_weighted_time.parquet")
    z.write("df_advisors_weighted_time.csv")

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>